In [19]:
import json
import os
from pathlib import Path

from pydantic import BaseModel, Field
from typing import Literal

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

print(project_root)

C:\Users\Administrator\Documents\ai-study\industrial-wastewater-ai


In [20]:
rules_data = [
    {
        "rule_id": "WQ-001",
        "title": "水质超标处理与价格协商机制",
        "category": "水质管理",
        "risk_level": "high",
        "review_question": (
            "合同是否明确水质超标时的处理方式、收费机制，"
            "以及严重超标情况下的价格协商机制？"
        ),
        "expected_requirement": (
            "合同应明确超标污水的拒收或接收条件、超标处理费用，"
            "以及现有计价规则不适用时的协商处理机制。"
        ),
        "suggestion": (
            "建议明确严重超标或特殊水质情况下，"
            "双方可就处理价格另行书面协商。"
        )
    }
]

rules_dir = project_root / "data" / "rules"
rules_dir.mkdir(parents=True, exist_ok=True)

rules_path = rules_dir / "contract_rules.json"

if not rules_path.exists():
    with open(rules_path, "w", encoding="utf-8") as f:
        json.dump(rules_data, f, ensure_ascii=False, indent=2)

print("规则文件准备完成")

规则文件准备完成


In [21]:
with open(rules_path, "r", encoding="utf-8") as f:
    rules = json.load(f)

rule = rules[0]

print(type(rules))
print(len(rules))
print(rule["title"])

<class 'list'>
1
水质超标处理与价格协商机制


In [22]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(project_root / ".env")

api_key = os.getenv("SILICONFLOW_API_KEY")

if not api_key:
    raise ValueError("未读取到 SILICONFLOW_API_KEY")

llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
    model="deepseek-ai/DeepSeek-V3.2"
)

In [23]:
class ContractReviewResult(BaseModel):

    rule_id: str = Field(
        description="审查规则编号"
    )

    status: Literal[
        "符合要求",
        "存在问题",
        "无法判断"
    ] = Field(
        description="依据当前提供的合同文本作出的审查判断"
    )

    evidence: str = Field(
        description="支持审查结论的合同原文"
    )

    issue: str = Field(
        description="发现的问题及其具体原因"
    )

    suggestion: str = Field(
        description="针对问题提出的合同修改建议"
    )

In [24]:
contract_text = """
6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，
乙方有权按照原污水处理单价的1.5倍收取超标处理费用。
"""

In [25]:
good_contract = """
6.3 水质超标处理费

甲方交付的污水超过接收标准，但超标倍数不超过2倍，
且经乙方评估确认可以处理的，
经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。

当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，
乙方有权拒绝接收。

如双方拟继续接收该类严重超标污水，
须在接收前另行书面协商处理价格。
未能达成一致的，乙方有权拒收。
"""

In [26]:
prompt = f"""
你是一名工业污水处理合同审查助手。

请严格依据以下审查规则，检查提供的合同条款。

审查规则编号：
{rule['rule_id']}

审查规则：
{rule['review_question']}

审查要求：
{rule['expected_requirement']}

合同原文：
{contract_text}

请回答：
1. 该条款是否满足审查要求？
2. 找出存在问题的原文。
3. 解释具体风险。
4. 给出修改建议。

注意：
1. 只根据提供的合同片段审查。
2. 没有出现的条款不能认定整份合同一定缺失。
3. 不得编造合同内容。
4. status只能是：符合要求、存在问题、无法判断。
5. evidence必须是字符串，直接引用原文。
6. issue和suggestion必须是字符串。
7. 如果符合要求，issue填写无，suggestion填写无需修改。
8. 必须返回rule_id，且与审查规则编号一致。
"""

In [27]:
structured_llm = llm.with_structured_output(
    ContractReviewResult
)

In [28]:
debug_llm = llm.with_structured_output(
    ContractReviewResult,
    include_raw=True
)

In [29]:
bad_debug = debug_llm.invoke(prompt)

print("解析结果：", bad_debug["parsed"])
print("解析错误：", bad_debug["parsing_error"])

解析结果： rule_id='WQ-001' status='存在问题' evidence='甲方交付的污水水质超过本协议约定的接收标准时，\n乙方有权按照原污水处理单价的1.5倍收取超标处理费用。' issue='条款仅约定了超标时的收费机制（按原单价1.5倍计费），但未明确针对超标污水的拒收或接收条件，也未约定在严重超标（如现有计价规则可能不适用时）情况下的协商处理机制。这可能导致在实际处理严重超标污水时，双方对是否接收、如何计价产生争议，增加履约风险。' suggestion='建议修改该条款，明确水质超标时的处理流程、收费机制及价格协商机制。例如：\n1. 明确超标污水的接收与拒收条件，如约定一定范围内的超标可接收并计收超标处理费，超出特定限值（如双方另行约定）乙方有权拒收或要求甲方进行预处理。\n2. 保留按原单价1.5倍收取超标处理费的规定，并增加当水质严重超标导致处理成本显著增加或现有计价规则明显不适用时，双方应另行协商处理费用或调整收费标准；协商不成的，可约定适用合同解除、违约责任等条款。\n修改示例：\n“6.3 水质超标处理\n6.3.1 甲方交付的污水水质超过本协议约定的接收标准但在[具体数值或倍数]范围内的，乙方有权接收并按原污水处理单价的1.5倍收取超标处理费用。\n6.3.2 若甲方交付的污水水质超过第6.3.1条约定的范围，或虽未超过但乙方认为现有收费机制明显不足覆盖其处理成本的，乙方有权要求甲方进行预处理或协商调整处理费用；协商不成的，乙方有权暂停接收该部分污水，且不免除甲方应支付的污水处理费用。具体协商机制及费用调整原则由双方另行签订补充协议确定。”'
解析错误： None


In [30]:
good_prompt = prompt.replace(
    contract_text,
    good_contract
)

good_debug = debug_llm.invoke(good_prompt)

print("解析结果：", good_debug["parsed"])
print("解析错误：", good_debug["parsing_error"])

解析结果： rule_id='WQ-001' status='符合要求' evidence='6.3 水质超标处理费\n甲方交付的污水超过接收标准，但超标倍数不超过2倍，\n且经乙方评估确认可以处理的，\n经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。\n\n当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，\n乙方有权拒绝接收。\n\n如双方拟继续接收该类严重超标污水，\n须在接收前另行书面协商处理价格。\n未能达成一致的，乙方有权拒收。' issue='无' suggestion='无需修改'
解析错误： None


In [31]:
def review_contract_rule(
    contract_text: str,
    rule: dict,
    structured_llm
) -> ContractReviewResult:

    prompt = f"""
    你是一名工业污水处理合同审查助手。

    审查规则编号：{rule['rule_id']}
    审查规则名称：{rule['title']}

    审查问题：
    {rule['review_question']}

    审查要求：
    {rule['expected_requirement']}

    参考建议：
    {rule['suggestion']}

    合同原文：
    {contract_text}

    请严格按照以下要求审查：

    1. 只依据提供的合同文本判断。
    2. 不得编造合同内容。
    3. status只能是：
       符合要求、存在问题、无法判断。
    4. evidence必须引用合同原文。
    5. 必须返回正确的rule_id。
    6. 不得自行编造具体价格、超标倍数或公司制度。
    7. 新增商务建议必须明确标注为待双方确认。

    请返回完整的结构化审查结果。
    """

    result = structured_llm.invoke(prompt)

    if result.rule_id != rule["rule_id"]:
        raise ValueError("返回的规则编号不匹配")

    return result

In [32]:
bad_result = review_contract_rule(
    contract_text,
    rule,
    structured_llm
)

good_result = review_contract_rule(
    good_contract,
    rule,
    structured_llm
)

print(bad_result.model_dump())
print(good_result.model_dump())

{'rule_id': 'WQ-001', 'status': '存在问题', 'evidence': '6.3 水质超标处理费\n\n甲方交付的污水水质超过本协议约定的接收标准时，\n乙方有权按照原污水处理单价的1.5倍收取超标处理费用。', 'issue': '合同条款仅规定了常规超标情形下的1.5倍费用计收标准，但未明确约定超过合同接收标准的污水是接收还是拒收。同时，对于严重超标或特殊水质导致现有计价规则（即1.5倍单价）可能不适用的情况，未设定双方另行协商的机制。', 'suggestion': '建议在合同中增加：1）甲方污水水质超标时的接收或拒收条件；2）明确约定若出现严重超标或特殊水质情况，致使本合同约定的1.5倍超标处理费用标准无法覆盖处理成本的，双方应本着公平原则，就该部分超标污水的处理价格另行书面协商。【待双方确认】'}
{'rule_id': 'WQ-001', 'status': '符合要求', 'evidence': '6.3 水质超标处理费\n甲方交付的污水超过接收标准，但超标倍数不超过2倍，\n且经乙方评估确认可以处理的，\n经乙方书面同意接收后，按照原污水处理单价的1.5倍收费。\n\n当任一指标超标超过2倍，或可能影响污水处理系统安全稳定运行时，\n乙方有权拒绝接收。\n\n如双方拟继续接收该类严重超标污水，\n须在接收前另行书面协商处理价格。\n未能达成一致的，乙方有权拒收。', 'issue': '合同明确规定了超标污水的拒收条件、接收条件、超标处理费用以及严重超标时的价格协商机制。', 'suggestion': '【待双方确认】建议在合同履行过程中，针对需启动另行协商的情况，约定明确的协商启动时限、协商期限及书面确认程序，以提升可操作性。'}


In [33]:
print(rules)

[{'rule_id': 'WQ-001', 'title': '水质超标处理与价格协商机制', 'category': '水质管理', 'risk_level': 'high', 'review_question': '合同是否明确水质超标时的处理方式、收费机制，以及严重超标情况下的价格协商机制？', 'expected_requirement': '合同应明确超标污水的拒收或接收条件、超标处理费用，以及现有计价规则不适用时的协商处理机制。', 'suggestion': '建议明确严重超标或特殊水质情况下，双方可就处理价格另行书面协商。'}]


In [34]:
new_rules = [
    {
        "rule_id": "WQ-002",
        "title": "水量计量与结算机制",
        "category": "费用结算",
        "risk_level": "high",
        "review_question": "合同是否明确污水接收水量的计量方式、计量点位、抄表周期、双方确认方式及计量争议处理机制？",
        "expected_requirement": "合同应明确计量设备及计量点位、水量记录和双方确认方式、结算周期，以及计量异常或争议时的复核机制。",
        "suggestion": "建议明确计量依据、共同抄表确认程序和计量争议处理方式。"
    },
    {
        "rule_id": "WQ-003",
        "title": "水质采样与检测机制",
        "category": "水质管理",
        "risk_level": "high",
        "review_question": "合同是否明确水质采样位置、采样频次、检测指标、检测方法、超标判定依据及检测结果争议处理机制？",
        "expected_requirement": "合同应明确采样点位、检测频次、检测指标和方法、超标判定依据，以及双方对检测结果存在异议时的复检程序。",
        "suggestion": "建议明确采样记录、检测结果确认、留样及异议复检机制。"
    }
]


In [36]:
existing_ids = {r['rule_id'] for r in rules}
rules.extend(
    r for r in new_rules
    if r['rule_id'] not in existing_ids
)
with open(rules_path,'w',encoding='utf-8') as f:
    json.dump(rules,f,ensure_ascii=False,indent=2)
print(len(rules))

3


In [40]:
rule_2 = next(r for r in rules
            if r['rule_id'] == 'WQ-002')

In [41]:
contract_2 = """
5.1 水量结算

污水处理费用按照实际接收水量进行结算，
甲方应按月支付污水处理费用。
"""

In [42]:
result_2 = review_contract_rule(
    contract_2,
    rule_2,
    structured_llm
)

In [43]:
print(result_2.model_dump())

{'rule_id': 'WQ-002', 'status': '存在问题', 'evidence': '污水处理费用按照实际接收水量进行结算，甲方应按月支付污水处理费用。', 'issue': '原文仅提及费用按‘实际接收水量’结算及‘按月支付’，但未约定水量的具体计量方式、计量点位、抄表周期、双方确认方式及计量争议处理机制。', 'suggestion': '待双方确认：建议在合同中补充明确：1.计量所依据的设备、安装位置（计量点位）；2.水量记录和核对的责任方、频率（如每日/每周共同抄表确认）；3.计量数据异常或双方对数据有争议时的复核与处理机制（如第三方检测、设备校验等）。'}


In [44]:
contract_3 = """
5.1 水质检测

乙方定期对甲方排放的污水进行水质检测，
检测结果作为污水处理费用结算的参考依据。
"""
rule_3 = next(
    r for r in rules
    if r["rule_id"] == "WQ-003"
)
result_3 = review_contract_rule(
    contract_3,
    rule_3,
    structured_llm
)

print(result_3.model_dump())

{'rule_id': 'WQ-003', 'status': '存在问题', 'evidence': "合同原文仅包含：'5.1 水质检测 乙方定期对甲方排放的污水进行水质检测，检测结果作为污水处理费用结算的参考依据。'", 'issue': '合同未明确采样位置、频次、指标、方法、超标判定及争议处理机制。', 'suggestion': '待双方确认：建议在合同条款中补充明确采样点位（如车间排放口、总排放口）、采样频次（如每日、每周）、检测指标（如COD、氨氮等）、检测方法（如参照国家标准方法）、超标判定依据（如约定的限值或国家标准）以及检测结果争议处理机制（如共同复检或委托第三方仲裁机构检测）。同时，建议增加采样记录双方签字确认、检测结果报告流程、超标样品/平行样品的留样保管及复检程序等细节，以增强合同的可执行性和公平性。'}


In [45]:
import docx

In [46]:
print(docx.__version__)

1.2.0


In [47]:
from docx import Document

In [49]:
doc = Document()
doc.add_heading("污水处理服务协议", level=1)

doc.add_heading("6.3 水质超标处理费", level=2)

doc.add_paragraph(
    "甲方交付的污水水质超过本协议约定的接收标准时，"
    "乙方有权按照原污水处理单价的1.5倍收取超标处理费用。"
)

doc.add_heading("附件六：污水处理费用", level=2)
table = doc.add_table(rows=2, cols=2)

table.cell(0, 0).text = "项目"
table.cell(0, 1).text = "单价"

table.cell(1, 0).text = "污水处理费"
table.cell(1, 1).text = "25元/吨"
test_dir = project_root / "data" / "raw" / "contracts"
test_dir.mkdir(parents=True, exist_ok=True)
test_path = test_dir / "test_contract.docx"
doc.save(test_path)
print("文件创建成功：", test_path)
print("文件是否存在：", test_path.exists())

文件创建成功： C:\Users\Administrator\Documents\ai-study\industrial-wastewater-ai\data\raw\contracts\test_contract.docx
文件是否存在： True


In [50]:
from docx import Document
doc = Document(test_path)
print("正文数量：",len(doc.paragraphs))

正文数量： 4


In [52]:
for i,paragraph in enumerate(doc.paragraphs):
    if paragraph.text.strip():
        print(i,paragraph.text)

0 污水处理服务协议
1 6.3 水质超标处理费
2 甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。
3 附件六：污水处理费用


In [53]:
print(len(doc.tables))

1


In [54]:
for table_index,table in enumerate(doc.tables):
    print(f'\n第{table_index+1}个表格')
    for row in table.rows:
        row_date = [cell.text.strip() for cell in row.cells]
        print(row_date)


第1个表格
['项目', '单价']
['污水处理费', '25元/吨']


In [60]:
from docx.text.paragraph import Paragraph
from docx.table import Table

contract_blocks = []
for block in doc.iter_inner_content():
    if isinstance(block,Paragraph):
        text = block.text.strip()
        if text:
            contract_blocks.append(text)
    elif isinstance(block,Table):
        table_rows = []
        for row in block.rows:
            cells = [
                cell.text.strip()
                for cell in row.cells
            ] 
            table_rows.append("|".join(cells))
        table_text = "\n".join(table_rows)
        contract_blocks.append('合同表格\n'+table_text)
contract_text = '\n\n'.join(contract_blocks)
print(contract_text)

污水处理服务协议

6.3 水质超标处理费

甲方交付的污水水质超过本协议约定的接收标准时，乙方有权按照原污水处理单价的1.5倍收取超标处理费用。

附件六：污水处理费用

合同表格
项目|单价
污水处理费|25元/吨
